# XLR8 ball demo
Renders a bouncing ball on the GPU in the FPGA fabric through `/dev/xlr8` and shows it here.
Run the loader first (`tools/board/demo.sh` does it): the bitstream, overlay and driver must be loaded, and the notebook kernel needs permission on `/dev/xlr8` (the demo script makes it world-writable).

In [ ]:
import sys, io
sys.path.insert(0, '/home/xilinx/xlr8/sw/lib')   # xlr8.py
import xlr8
from PIL import Image
from IPython.display import Image as IPImage, display

W = H = 64
SRC, DST = 0x2000, 0x6000
BALL, BG = 0x00FF8000, 0x00101830

def frames(n, radius=7, x=10*16, y=14*16, vx=21, vy=13):
    # same integer physics as tools/animate_ref.py
    lo_x, hi_x, lo_y, hi_y = radius*16, (W-1-radius)*16, radius*16, (H-1-radius)*16
    for _ in range(n):
        yield x >> 4, y >> 4, radius*radius
        x += vx; y += vy
        if x < lo_x: x, vx = 2*lo_x - x, -vx
        elif x > hi_x: x, vx = 2*hi_x - x, -vx
        if y < lo_y: y, vy = 2*lo_y - y, -vy
        elif y > hi_y: y, vy = 2*hi_y - y, -vy


In [ ]:
imgs, cycles = [], []
with xlr8.Device() as gpu:
    print(gpu.info())
    gpu.load_file('/home/xilinx/xlr8/ball.hex')
    gpu.alloc(0x20000)
    for cx, cy, r2 in frames(60):
        gpu.words[SRC:SRC+5] = [cx & 0xFFFFFFFF, cy & 0xFFFFFFFF, r2, BALL, BG]
        res = gpu.run(threads=W*H, src=SRC, dst=DST)
        assert res.ok, res
        px = gpu.words[DST:DST+W*H]
        im = Image.new('RGB', (W, H))
        im.putdata([((v >> 16) & 255, (v >> 8) & 255, v & 255) for v in px])
        imgs.append(im.resize((256, 256), Image.NEAREST))
        cycles.append(res.cycles)
print('mean GPU cycles per frame:', sum(cycles)//len(cycles), '=', round(sum(cycles)/len(cycles)*20e-6, 2), 'ms at 50 MHz')
buf = io.BytesIO()
imgs[0].save(buf, format='GIF', save_all=True, append_images=imgs[1:], duration=40, loop=0)
display(IPImage(data=buf.getvalue()))
